# Mission Profile
### Roadrunner propulsion reverse-engineering study · notebook 2 of 15

**Purpose.** The sample mission is an *input* (ADR-0018): its shape follows what the
manufacturer has said, its numbers are assumptions. This notebook quotes the evidence
each segment rests on, lays the profile out in time, and plots it. It answers:

> *What exactly is the mission the rest of the study flies, what in it is evidence,
> and what is assumed?*

It then lists the performance requirements that go with the mission, which the
constraint analysis will turn into curves (ADR-0019, ADR-0027).

It flies nothing: no weights, no fuel, no engine, no constraint curves. Those are the
mission and constraint analyses, coupled later.

---
## 0 · Setup

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent / "src"))   # bare-checkout fallback

from roadrunner_re import IsaAtmosphere, load_claims, load_mission, reports
from roadrunner_re import constraint_inputs as ci
from roadrunner_re import mission_timeline as mt
from roadrunner_re.notebook import nb_setup
from roadrunner_re.plots.mission import mission_profile_figure

paths = nb_setup()
claims = load_claims()
mission = load_mission(claims=claims)
requirements = ci.load_constraints(claims=claims, mission=mission)
atmosphere = IsaAtmosphere.from_yaml()

---
## 1 · What the profile rests on

Each segment is quoted with the ledger statements it cites and its stated assumption.
The tier is shown: **A** stated by the manufacturer, **B** secondary report. Tier-X
claims are never inputs, and the loader refuses a mission that cites one. Every claim
here is qualitative: nobody has published a speed, altitude, range or endurance.

In [ ]:
print(reports.mission_quotes(mission, claims))

---
## 2 · The numbers

Every figure is nominal with its low and high bound in brackets; all are sampled in the
uncertainty analysis (ADR-0020). The day is hot (ISA + the offset below) for the whole
mission, because vertical launch on a hot day is the engine sizing case (ADR-0009).

A dash means the profile does not fix the figure. Climb and descent durations are one
such case: they follow from the thrust margin and are found in the mission analysis,
not assumed here.

Dash and cruise are given as distances, so their durations are distance divided by
true airspeed on the hot day, $t = s / (M\,a)$, with the speed of sound $a$ from the
one standard atmosphere (ADR-0017). Bounds come from the corners of the input bounds.

In [ ]:
views = mt.mission_timeline(mission, atmosphere)
print(f"day: ISA {mission.dT_isa_K.value:+.0f} K ({mission.dT_isa_K.low:+.0f} to {mission.dT_isa_K.high:+.0f}), "
      f"payload {mission.payload_mass_kg.value:.0f} kg ({mission.payload_mass_kg.low:.0f}-{mission.payload_mass_kg.high:.0f}), "
      f"reserve {mission.reserve_fraction.value:.0%} of mission fuel\n")
print(reports.mission_table(views))

---
## 3 · The profile

In [ ]:
fig = mission_profile_figure(views, paths.figures / "mission_profile.png")

Segments drawn to scale are those whose duration the profile fixes; the hatched ones
(climb, descent) are drawn at an arbitrary width. The shaded bands are the input bounds,
not a forecast.

---
## 4 · The requirements that go with it

Each requirement bounds the thrust loading $T_{SL}/W_{TO}$ (ADR-0019). **Lower bound:**
the engines must give at least this much. **Upper bound:** they must not exceed it; the
landing thrust floor is the only one, because an engine that cannot throttle down far
enough cannot hover the vehicle onto the ground.

Where a requirement belongs to a mission segment, its Mach number, altitude and the day
are *taken from the mission* (marked `*`), not repeated, so changing the mission moves
the requirements with it. `weight at` names the segment at whose start the weight fraction
$\beta$ is evaluated; the mission analysis supplies it. The numbers owned by the
requirements are assumptions with wide bounds, like the mission's.

In [ ]:
print(reports.constraint_table(requirements))

Evidence and assumptions behind each requirement:

In [ ]:
print(reports.constraint_quotes(requirements, claims))

---
## 5 · Handoff

In [ ]:
content = mt.mission_profile_handoff(mission, views)
content["constraints"] = ci.constraints_handoff(requirements)
content["uncertain_inputs"] = sorted(list(content["uncertain_inputs"]) + list(requirements.uncertain_inputs()))
handoff = mt.write_mission_profile_handoff(content, paths.out / "mission_profile.yaml")
print(f"wrote {handoff.relative_to(paths.repo)}")

---
## Findings

- **Nothing in the profile is measured.** Its shape comes from qualitative statements
  (vertical launch and landing, high-subsonic transit, loiter, return); every number is
  an assumption with wide bounds, and the least supported one is the payload mass.
- **The loiter dominates the time.** Dash and return together take a few minutes; the
  loiter is assumed to last a quarter of an hour, with bounds from five to thirty
  minutes. Its Mach number is lower than the dash, so it flies at part throttle.
- **The vertical segments are short but set the engine.** Launch and landing last
  seconds to a minute, yet they set the thrust requirement (ADR-0009).
- **Standing finding:** climb and descent durations are not fixed by the profile, so the
  total mission time is not known until the mission analysis exists.
- **Only one requirement is a statement about thrust above weight, and it is an assumption.**
  The vertical launch margin sets the engine size, yet no source gives it; the landing floor
  is the only upper bound and checks the engines' published idle thrust.
- **The cruise-type requirements add no new numbers.** Dash is thrust equals drag at the
  mission's dash condition; the others (climb, manoeuvre, loiter ceiling) are placeholders
  expected not to bind, which the constraint analysis will confirm or refute.